# 🧋 Llama-3.2-3B × TaiwanChat — QLoRA 繁中/台灣語感微調

**Built with Llama**

用 [Unsloth](https://github.com/unslothai/unsloth) 以 QLoRA 微調 Llama-3.2-3B-Instruct,資料集為 [yentinglin/TaiwanChat](https://huggingface.co/datasets/yentinglin/TaiwanChat) 子集,目標是更自然的繁體中文與台灣在地語感。程式碼與完整說明:[github.com/tun0000/llama32-taiwanchat-qlora](https://github.com/tun0000/llama32-taiwanchat-qlora)

**執行三步驟:**
1. 選單 **Runtime(執行階段)→ Change runtime type → L4 GPU**(建議;T4 也可,約慢 3 倍)。
2. 左側 **🔑 Secrets → Add new secret**:名稱 `HF_TOKEN`、值為 HuggingFace **write** token([hf.co/settings/tokens](https://huggingface.co/settings/tokens)),並打開「Notebook access」開關。
3. 下方參數 cell 保持 `SMOKE_TEST = True`,**Runtime → Run all** 先跑通一次(約 10 分鐘);成功後改 `False` 再 Run all 正式訓練(L4 約 1–2 小時)。

> 訓練資料 TaiwanChat 為 CC BY-NC 4.0 → 產出模型僅供研究/非商業用途。


In [ ]:
# ============ 參數(只需要改這裡) ============
BASE_MODEL      = "unsloth/Llama-3.2-3B-Instruct"    # meta-llama/Llama-3.2-3B-Instruct 的 ungated 鏡像(權重相同,免申請)
# BASE_MODEL    = "meta-llama/Llama-3.2-3B-Instruct"        # 官方 gated 版:需先在 HF 模型頁接受授權並獲核准
# BASE_MODEL    = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"  # 預量化 4bit:下載最快
# BASE_MODEL    = "unsloth/Meta-Llama-3.1-8B-Instruct"      # 升級 8B(Llama-3.2 沒有 8B;3.1-8B 用同一套 chat template)
DATASET_ID      = "yentinglin/TaiwanChat"
INCLUDE_SOURCES = None         # None = 全部 9 種資料源;消融實驗 v2:["n/a", "sharegpt", "gpt4"](理由見 README)
N_SAMPLES       = 15000        # 訓練樣本數(過長樣本過濾前)
MAX_SEQ_LENGTH  = 2048
LORA_R          = 16
LORA_ALPHA      = 16
LEARNING_RATE   = 2e-4
NUM_EPOCHS      = 1
MAX_STEPS       = None         # None → 以 NUM_EPOCHS 為準;填 int 則覆蓋 epoch 設定
SEED            = 3407
MAX_NEW_TOKENS  = 256          # 前後對照推論的生成長度

SMOKE_TEST      = True         # ★ 先 True 跑通整條流程(~10 分鐘),再改 False 正式訓練
PUSH_TO_HUB     = "auto"       # "auto" → SMOKE_TEST 時不推、正式時推;也可直接 True / False

HF_USERNAME       = ""         # 留空 → 自動用 HF token 對應的帳號
REPO_BASE         = ""         # 留空 → 由 BASE_MODEL 自動推導(例:llama-3.2-3b-taiwan-chat)
RUN_TAG           = ""         # 實驗標籤,附加在 repo 名上;消融實驗 v2 用 "-clean"

# ---- 由上面推導(不用改)。SMOKE_TEST 的效果全部集中在這裡,其他 cell 沒有任何分支 ----
if SMOKE_TEST:
    N_SAMPLES, MAX_STEPS = 200, 20
PUSH_TO_HUB   = (not SMOKE_TEST) if PUSH_TO_HUB == "auto" else bool(PUSH_TO_HUB)
LOGGING_STEPS = 1 if SMOKE_TEST else 10
if not REPO_BASE:   # "unsloth/Meta-Llama-3.1-8B-Instruct" → "llama-3.1-8b-taiwan-chat"
    REPO_BASE = (BASE_MODEL.split("/")[-1].lower()
                 .replace("meta-", "").replace("-instruct", "").replace("-bnb-4bit", "")) + "-taiwan-chat"
ADAPTER_REPO_NAME = f"{REPO_BASE}{RUN_TAG}-lora"
MERGED_REPO_NAME  = f"{REPO_BASE}{RUN_TAG}"
print(f"SMOKE_TEST={SMOKE_TEST}  N_SAMPLES={N_SAMPLES}  MAX_STEPS={MAX_STEPS}  PUSH_TO_HUB={PUSH_TO_HUB}")
print(f"INCLUDE_SOURCES={INCLUDE_SOURCES}  →  repo: {ADAPTER_REPO_NAME} / {MERGED_REPO_NAME}")


In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2


In [ ]:
# 環境檢查:上面的安裝 cell 有 %%capture(安靜),這裡響亮地驗證一切就緒
import locale
locale.getpreferredencoding = lambda *args: "UTF-8"   # 已知 Colab locale bug 的 workaround

import torch
assert torch.cuda.is_available(), (
    "偵測不到 GPU!請點 Runtime → Change runtime type → 選 L4(建議)或 T4,再 Runtime → Run all。"
)
import unsloth            # 必須先 import unsloth,讓它 patch transformers/trl
import transformers, trl, datasets, peft
GPU_NAME = torch.cuda.get_device_name(0)
BF16_OK  = torch.cuda.is_bf16_supported()
print(f"GPU = {GPU_NAME}(bf16 {'支援' if BF16_OK else '不支援 → 自動改用 fp16'})")
print(f"torch={torch.__version__}  transformers={transformers.__version__}  trl={trl.__version__}")
print(f"datasets={datasets.__version__}  peft={peft.__version__}  unsloth={unsloth.__version__}")


In [ ]:
# HF 登入 + 推送預檢:權限問題要在「第 4 個 cell」爆,而不是訓練兩小時之後
from google.colab import userdata
from huggingface_hub import HfApi, login

HF_TOKEN = None
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

if PUSH_TO_HUB or BASE_MODEL.startswith("meta-llama/"):
    assert HF_TOKEN, (
        "讀不到 HF_TOKEN!請點左側 🔑(Secrets)→ Add new secret:名稱 HF_TOKEN、"
        "值為 HuggingFace write token(hf.co/settings/tokens),打開 Notebook access 後重跑本 cell。"
    )

api = HfApi(token=HF_TOKEN)
if HF_TOKEN:
    login(token=HF_TOKEN)
    HF_USERNAME = HF_USERNAME or api.whoami()["name"]

ADAPTER_REPO = f"{HF_USERNAME}/{ADAPTER_REPO_NAME}" if HF_USERNAME else None
MERGED_REPO  = f"{HF_USERNAME}/{MERGED_REPO_NAME}"  if HF_USERNAME else None

if PUSH_TO_HUB:
    api.create_repo(ADAPTER_REPO, exist_ok=True)   # 現在就建 repo:token 沒有 write 權限會立刻爆
    api.create_repo(MERGED_REPO,  exist_ok=True)
    print(f"訓練後將推送到:\n  https://huggingface.co/{ADAPTER_REPO}\n  https://huggingface.co/{MERGED_REPO}")
if BASE_MODEL.startswith("meta-llama/"):
    api.model_info(BASE_MODEL)   # 還沒接受 Llama 3.2 授權會在這裡 401/403:請到模型頁申請並等核准
print(f"HF_USERNAME = {HF_USERNAME or '(未登入;SMOKE_TEST 不推送可不設 token)'}")


In [ ]:
# %%DATA_PREP_MODULE%%  AUTOGENERATED from scripts/data_prep.py — 不要直接改這個 cell!
# 請改 scripts/data_prep.py,然後執行:python scripts/sync_notebook.py
# sha256(scripts/data_prep.py) = b92e07d0b881272784ec200296bc1f70df5213000885f6d4eb53a6c1c8f52d14
# -*- coding: utf-8 -*-
"""TaiwanChat 資料處理的「單一事實來源」(single source of truth)。

這個模組同時被兩邊使用:
  1. scripts/preview_dataset.py —— 本機 CPU 驗證(plain transformers AutoTokenizer)
  2. llama32-taiwanchat-qlora.ipynb —— Colab GPU 訓練(unsloth 修補過的 tokenizer)

notebook 裡的副本由 scripts/sync_notebook.py 自動注入;改完本檔請執行:
    python scripts/sync_notebook.py
用 `python scripts/sync_notebook.py --check` 可偵測兩邊是否漂移。

刻意只用 Python 標準函式庫(不 import torch / transformers / datasets / unsloth):
tokenizer 與 dataset 都由呼叫端注入,因此在沒有 GPU、沒裝 unsloth 的環境也能執行。
"""

import hashlib
from collections import Counter

DATASET_ID = "yentinglin/TaiwanChat"

# Llama-3.x chat template 會在 system header 塞入「Today Date」;固定日期
# (meta 官方 template 的預設值)讓本機與 Colab 的輸出完全可重現、可比對。
FIXED_DATE = "26 Jul 2024"

# ShareGPT 欄位("from")→ OpenAI 欄位("role")的對應。
ROLE_MAP = {
    "human": "user",
    "gpt": "assistant",
    "system": "system",
    # 已是 OpenAI 風格時原樣通過
    "user": "user",
    "assistant": "assistant",
}

# train_on_responses_only 依賴的 Llama-3.x header 標記(結尾的 \n\n 是必要的)。
LLAMA3_USER_MARKER = "<|start_header_id|>user<|end_header_id|>\n\n"
LLAMA3_ASSISTANT_MARKER = "<|start_header_id|>assistant<|end_header_id|>\n\n"

# 微調前後對照用的 5 個台灣情境問題(notebook 與 model card 共用)。
TAIWAN_TEST_QUESTIONS = [
    "請用台灣人習慣的說法介紹珍珠奶茶:它的由來、基本做法,以及為什麼能代表台灣的手搖飲文化。",
    "我是剛來台灣的交換學生,想辦手機門號。請比較預付卡和月租方案的差別,以及辦理時要準備哪些證件。",
    "請推薦台南有名的夜市,並列出五樣必吃的小吃,每樣用一兩句話介紹。",
    "「土豆」這個詞在台灣和中國大陸分別指什麼?請再舉三組兩岸說法不同的日常用語,整理成表格。",
    "請解釋台灣的「垃圾不落地」政策,以及為什麼垃圾車會播放音樂?",
]


def count_sources(dataset):
    """回傳 {id 資料源: 筆數},由多到少。TaiwanChat 有 9 種 id(flan/sharegpt/self/…)。"""
    return dict(Counter(dataset["id"]).most_common())


def filter_by_sources(dataset, include_sources):
    """依 `id` 欄過濾資料源(消融實驗用)。include_sources=None → 原樣返回。

    回傳 (dataset, stats dict)。與 notebook、preview 共用,確保兩邊過濾語意一致。
    """
    if include_sources is None:
        return dataset, {"filtered": False}
    include = set(include_sources)
    n_before = len(dataset)
    ds = dataset.filter(lambda ex: ex["id"] in include)
    return ds, {
        "filtered": True,
        "include": sorted(include),
        "n_before": n_before,
        "n_after": len(ds),
    }


def normalize_text(text):
    """統一換行(資料裡混有 Windows \\r\\n)並去除頭尾空白。"""
    return (text or "").replace("\r\n", "\n").strip()


def standardize_sharegpt(conversation):
    """把 ShareGPT 格式 [{"from": "human"/"gpt", "value": ...}] 轉成
    OpenAI 格式 [{"role": "user"/"assistant", "content": ...}]。

    語意等同 unsloth.chat_templates.standardize_sharegpt,但不依賴 unsloth,
    因此本機 CPU 也能跑同一套邏輯。未知 role 會丟 ValueError(呼叫端決定丟棄)。
    """
    messages = []
    for turn in conversation:
        src_role = turn.get("from", turn.get("role"))
        role = ROLE_MAP.get(src_role)
        if role is None:
            raise ValueError("unknown role: %r" % (src_role,))
        content = normalize_text(turn.get("value", turn.get("content")))
        if not content:
            raise ValueError("empty message content")
        messages.append({"role": role, "content": content})
    if not messages:
        raise ValueError("empty conversation")
    return messages


def render_conversation(conversation, tokenizer):
    """standardize + 套用 tokenizer 的 Llama-3.x chat template,回傳純文字。

    date_string 固定為 FIXED_DATE,消除 template 內建「今天日期」造成的
    跨日期/跨環境非決定性。轉換失敗回傳 ""(由 prepare_dataset 過濾並計數)。
    """
    try:
        messages = standardize_sharegpt(conversation)
    except ValueError:
        return ""
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        date_string=FIXED_DATE,
    )


def formatting_prompts_func(examples, tokenizer):
    """datasets.map 用的 batched 函式:conversations → {"text": [...]}"""
    texts = [render_conversation(convo, tokenizer) for convo in examples["conversations"]]
    return {"text": texts}


def add_token_counts(examples, tokenizer):
    """datasets.map 用的 batched 函式:計算每筆 text 的 token 數。

    template 輸出已含 <|begin_of_text|>,所以 add_special_tokens=False,
    避免重複計入 BOS 導致過濾邊界偏移。
    """
    ids = tokenizer(examples["text"], add_special_tokens=False)["input_ids"]
    return {"n_tokens": [len(x) for x in ids]}


def prepare_dataset(dataset, tokenizer, max_seq_length):
    """完整資料處理管線:格式化 → 丟無效列 → 計 token → 過濾過長樣本。

    dataset:已由呼叫端取樣完成的 datasets.Dataset
      (notebook 端:shuffle(seed).select(range(N));preview 端:streaming 前 N 筆)。
    回傳 (只剩 "text" 欄的 dataset, 統計 dict)。
    """
    n_input = len(dataset)

    ds = dataset.map(
        formatting_prompts_func,
        batched=True,
        fn_kwargs={"tokenizer": tokenizer},
        remove_columns=[c for c in dataset.column_names if c != "text"],
    )
    ds = ds.filter(lambda ex: ex["text"] != "")
    n_valid = len(ds)

    ds = ds.map(add_token_counts, batched=True, fn_kwargs={"tokenizer": tokenizer})
    all_lens = sorted(ds["n_tokens"])
    ds = ds.filter(lambda ex: ex["n_tokens"] <= max_seq_length)
    n_kept = len(ds)
    ds = ds.remove_columns(["n_tokens"])

    stats = {
        "n_input": n_input,
        "n_dropped_invalid": n_input - n_valid,
        "n_dropped_overlong": n_valid - n_kept,
        "n_kept": n_kept,
        "max_seq_length": max_seq_length,
    }
    if all_lens:
        stats.update({
            "token_len_min": all_lens[0],
            "token_len_mean": round(sum(all_lens) / len(all_lens), 1),
            "token_len_p95": all_lens[min(len(all_lens) - 1, int(len(all_lens) * 0.95))],
            "token_len_max": all_lens[-1],
        })
    return ds, stats


def describe_dataset(dataset, n_rows=2, max_chars=300):
    """印出欄位 schema 與前 n_rows 筆(長字串截斷)。兩邊輸出格式一致,方便比對。"""

    def _trunc(value):
        if isinstance(value, str):
            return value if len(value) <= max_chars else value[:max_chars] + "…[截斷]"
        if isinstance(value, list):
            return [_trunc(v) for v in value]
        if isinstance(value, dict):
            return {k: _trunc(v) for k, v in value.items()}
        return value

    lines = ["欄位:%s" % (dataset.column_names,)]
    for i in range(min(n_rows, len(dataset))):
        lines.append("--- 第 %d 筆 ---" % i)
        for key, value in dataset[i].items():
            lines.append("%s = %r" % (key, _trunc(value)))
    return "\n".join(lines)


def crosscheck_messages_column(row):
    """用資料集內建的平行欄位 `messages`(OpenAI 格式)驗證我們的
    standardize_sharegpt(conversations) 轉換結果。回傳不一致描述清單(空 = OK)。"""
    problems = []
    try:
        ours = standardize_sharegpt(row["conversations"])
    except ValueError as e:
        return ["standardize failed: %s" % e]
    theirs = [
        {"role": m["role"], "content": normalize_text(m["content"])}
        for m in row["messages"]
    ]
    if len(ours) != len(theirs):
        return ["turn count mismatch: ours=%d dataset=%d" % (len(ours), len(theirs))]
    for i, (a, b) in enumerate(zip(ours, theirs)):
        if a["role"] != b["role"]:
            problems.append("turn %d role mismatch: %r vs %r" % (i, a["role"], b["role"]))
        if a["content"] != b["content"]:
            problems.append("turn %d content mismatch" % i)
    return problems


def check_llama3_markers(text):
    """確認 render 結果含有 response-only loss masking 依賴的 Llama-3 header 標記。"""
    missing = [m for m in (LLAMA3_USER_MARKER, LLAMA3_ASSISTANT_MARKER) if m not in text]
    if missing:
        raise AssertionError(
            "chat template 輸出缺少 Llama-3 header 標記 %r —— "
            "train_on_responses_only 會把所有 label 遮罩掉,等於白練!" % (missing,)
        )
    return True


def format_parity_sha256(tokenizer):
    """對固定探針對話 render 後取 SHA-256。

    本機 preview 與 Colab notebook 各印一次:相同 → 兩邊格式化 100% 一致;
    不同 → template 來源有差(例如 unsloth get_chat_template 與 stock template
    的日期行差異),需人工確認差異僅限於此。
    """
    probe = [
        {"role": "user", "content": "測試"},
        {"role": "assistant", "content": "回覆"},
    ]
    text = tokenizer.apply_chat_template(
        probe, tokenize=False, add_generation_prompt=False, date_string=FIXED_DATE
    )
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


In [ ]:
# 載入 TaiwanChat:先看清楚 schema 與實際樣本,再取子集
from datasets import load_dataset

raw_ds = load_dataset(DATASET_ID, split="train")
print(raw_ds)                                # 欄位 + 485,432 筆
print(describe_dataset(raw_ds, n_rows=2))    # 共用 helper:與本機 scripts/preview_dataset.py 輸出同格式

if INCLUDE_SOURCES is not None:              # 消融實驗:只留指定資料源
    print("\n資料源分佈(過濾前):", count_sources(raw_ds))
    raw_ds, src_stats = filter_by_sources(raw_ds, INCLUDE_SOURCES)
    print(f"依 INCLUDE_SOURCES={src_stats['include']} 過濾:{src_stats['n_before']} → {src_stats['n_after']} 筆")
assert len(raw_ds) >= N_SAMPLES, f"過濾後只剩 {len(raw_ds)} 筆,不夠 N_SAMPLES={N_SAMPLES}"

subset = raw_ds.shuffle(seed=SEED).select(range(N_SAMPLES))   # shuffle 再取,避免資料源排序偏差
print(f"\n取樣 {len(subset)} 筆(shuffle seed={SEED})")


In [ ]:
# 以 4bit 載入 base model(QLoRA)+ 套 LoRA
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,        # 自動:T4 → fp16、L4/A100 → bf16
    load_in_4bit   = True,        # QLoRA:NF4 4-bit 量化載入
    token          = HF_TOKEN,    # ungated 鏡像會忽略;meta-llama 需要
)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")   # "llama-3.2" 是上游同一 template 的 alias

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",   # 省 ~30% VRAM
    random_state = SEED,
    use_rslora = False,
    loftq_config = None,
)


In [ ]:
# 套用共用資料處理管線(與本機 scripts/preview_dataset.py 完全同一套程式碼)
train_ds, prep_stats = prepare_dataset(subset, tokenizer, MAX_SEQ_LENGTH)
print(prep_stats)
print("=" * 80)
print(train_ds[0]["text"][:1200])

check_llama3_markers(train_ds[0]["text"])   # markers 是 response-only masking 的依賴,缺了直接失敗
print("\nFORMAT_PARITY_SHA256 =", format_parity_sha256(tokenizer))
print("(應與本機 scripts/preview_dataset.py 印出的值一致;不一致的說明見 README)")


In [ ]:
# 「微調前」推論:LoRA 的 B 矩陣初始為 0 → 此刻 adapter 貢獻為 0,
# 輸出與 base model 完全相同,因此不必另外載一份模型當 baseline。
def generate_answers(model, tokenizer, questions, max_new_tokens=256):
    """Greedy decoding(do_sample=False):輸出確定,前後差異可完全歸因於權重變化。"""
    answers = []
    for q in questions:
        input_ids = tokenizer.apply_chat_template(
            [{"role": "user", "content": q}],
            tokenize=True, add_generation_prompt=True, return_tensors="pt",
            date_string=FIXED_DATE,
        ).to(model.device)
        out = model.generate(
            input_ids=input_ids, max_new_tokens=max_new_tokens,
            do_sample=False, temperature=None, top_p=None, repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id, use_cache=True,
        )
        answers.append(tokenizer.decode(out[0, input_ids.shape[1]:], skip_special_tokens=True).strip())
    return answers

FastLanguageModel.for_inference(model)
before_answers = generate_answers(model, tokenizer, TAIWAN_TEST_QUESTIONS, MAX_NEW_TOKENS)
for q, a in zip(TAIWAN_TEST_QUESTIONS, before_answers):
    print("Q:", q, "\nA:", a[:400], "\n" + "-" * 80)
FastLanguageModel.for_training(model)   # 重要:回到訓練模式再建 trainer


In [ ]:
# SFTTrainer(unsloth 修補過的 trl==0.22.2 寫法)+ 只對 assistant 回覆計 loss
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
from unsloth.chat_templates import train_on_responses_only
import torch

BSZ        = 4 if torch.cuda.is_bf16_supported() else 2   # L4 → 4、T4 → 2
GRAD_ACCUM = 8 // BSZ                                      # 有效 batch 固定 8,兩種 GPU 曲線可比

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),   # train_on_responses_only 必需
    packing = False,                                                  # train_on_responses_only 必需
    args = SFTConfig(
        per_device_train_batch_size = BSZ,
        gradient_accumulation_steps = GRAD_ACCUM,
        num_train_epochs = NUM_EPOCHS,
        max_steps = MAX_STEPS if MAX_STEPS is not None else -1,   # -1 → 以 epoch 為準
        learning_rate = LEARNING_RATE,
        warmup_ratio = 0.03,
        lr_scheduler_type = "linear",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        logging_steps = LOGGING_STEPS,
        seed = SEED,
        save_strategy = "no",       # 結尾直接 push,不留中間 checkpoint
        output_dir = "outputs",
        report_to = "none",
        # 刻意不設 fp16/bf16:unsloth 依 GPU 能力自動選
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = LLAMA3_USER_MARKER,        # "<|start_header_id|>user<|end_header_id|>\n\n"
    response_part    = LLAMA3_ASSISTANT_MARKER,   # "<|start_header_id|>assistant<|end_header_id|>\n\n"
)

# ---- masking 驗證:遮罩錯 → loss 全 0 → 白練。這裡直接把「計 loss 的 token」解碼出來看 ----
batch  = trainer.data_collator([trainer.train_dataset[0]])
labels = batch["labels"][0].tolist()
n_trained = sum(1 for l in labels if l != -100)
assert n_trained > 0, "所有 label 都被遮罩了!markers 與 chat template 不符。"
kept = [t for t, l in zip(batch["input_ids"][0].tolist(), labels) if l != -100]
print(f"樣本 0:{n_trained}/{len(labels)} 個 token 計入 loss。以下應該只有 assistant 回覆內容:")
print(tokenizer.decode(kept)[:600])


In [ ]:
trainer_stats = trainer.train()
print(trainer_stats.metrics)


In [ ]:
# loss 曲線
import matplotlib.pyplot as plt

hist = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
steps, losses = zip(*hist)
plt.figure(figsize=(8, 4))
plt.plot(steps, losses)
plt.xlabel("step"); plt.ylabel("training loss")
plt.title(f"{ADAPTER_REPO_NAME}  (SMOKE_TEST={SMOKE_TEST})")
plt.grid(alpha=0.3)
plt.savefig("loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# 「微調後」推論:同 5 題、同 greedy 參數 → 差異 100% 來自訓練
FastLanguageModel.for_inference(model)
after_answers = generate_answers(model, tokenizer, TAIWAN_TEST_QUESTIONS, MAX_NEW_TOKENS)

comparison = [
    {"question": q, "before": b, "after": a}
    for q, b, a in zip(TAIWAN_TEST_QUESTIONS, before_answers, after_answers)
]
for row in comparison:
    print("=" * 100)
    print("Q:", row["question"])
    print("--- 微調前 ---"); print(row["before"][:500])
    print("--- 微調後 ---"); print(row["after"][:500])


In [ ]:
# 產生兩份 model card(繁中為主 + 英文摘要)。用 string.Template:
# 卡片裡含 {"role": ...} 之類的程式片段,f-string 的大括號跳脫太容易出錯。
import time
from string import Template

RUN_DATE = time.strftime("%Y-%m-%d")

# Llama 3.1 / 3.2 各自是獨立的 Community License(條款相似但版本、連結不同),
# 依 BASE_MODEL 動態判斷,避免訓練 8B(Llama-3.1 系列)時誤標成 3.2 的授權。
_LLAMA_VER   = "3.1" if "3.1" in BASE_MODEL else "3.2"
LICENSE_SLUG = f"llama{_LLAMA_VER}"
LICENSE_NAME = f"Llama {_LLAMA_VER} Community License"
LICENSE_LINK = f"https://www.llama.com/llama{_LLAMA_VER.replace('.', '_')}/license/"
LICENSE_AUP  = f"https://www.llama.com/llama{_LLAMA_VER.replace('.', '_')}/use-policy"
NOTICE_TEXT  = (f"Llama {_LLAMA_VER} is licensed under the {LICENSE_NAME}, "
                "Copyright © Meta Platforms, Inc. All Rights Reserved.")

BASE_MODEL_DISPLAY = BASE_MODEL.split("/")[-1]   # 例:"Llama-3.2-3B-Instruct" / "Meta-Llama-3.1-8B-Instruct"
if BASE_MODEL.startswith("meta-llama/"):
    CANONICAL_REPO = BASE_MODEL
    MIRROR_NOTE = ""
else:
    CANONICAL_REPO = "meta-llama/" + BASE_MODEL.split("/")[-1].replace("-bnb-4bit", "")
    MIRROR_NOTE = f"({CANONICAL_REPO} 之權重相同鏡像)"
RUN_LABEL = f" ({RUN_TAG.lstrip('-')})" if RUN_TAG else ""   # 消融實驗標籤,例:" (clean)"

SMOKE_BANNER = ""
if SMOKE_TEST:
    SMOKE_BANNER = ("\n> ⚠️ **SMOKE TEST 產物 — 不是正式模型**:只用 200 筆資料訓練 20 步,僅供流程驗證。\n")

metrics      = trainer_stats.metrics
FINAL_LOSS   = f"{metrics.get('train_loss', float('nan')):.4f}"
RUNTIME_MIN  = f"{metrics.get('train_runtime', 0) / 60:.1f}"
TRAIN_AMOUNT = f"{NUM_EPOCHS} epoch" if MAX_STEPS in (None, -1) else f"{MAX_STEPS} steps"
_adapter_repo = ADAPTER_REPO or f"<HF_USERNAME>/{ADAPTER_REPO_NAME}"
_merged_repo  = MERGED_REPO  or f"<HF_USERNAME>/{MERGED_REPO_NAME}"

def _examples_md(comparison, limit=300):
    blocks = []
    for row in comparison:
        b = row["before"][:limit] + ("…" if len(row["before"]) > limit else "")
        a = row["after"][:limit]  + ("…" if len(row["after"])  > limit else "")
        blocks.append(
            "<details>\n<summary>" + row["question"] + "</summary>\n\n"
            "**微調前(base):**\n\n" + b + "\n\n**微調後(fine-tuned):**\n\n" + a + "\n\n</details>"
        )
    return "\n\n".join(blocks)

CARD_TEMPLATE = Template("""---
license: $license_slug
license_name: $license_slug
license_link: $license_link
language:
- zh
- en
base_model: $base_model
datasets:
- yentinglin/TaiwanChat
library_name: $library_name
pipeline_tag: text-generation
tags:
- llama-$llama_ver
- unsloth
- qlora
- lora
- taiwan
- traditional-chinese
- zh-tw
- sft
---

# $title

**Built with Llama**
$smoke_banner
$intro

*(English)* $en_summary

## 訓練設定

| 項目 | 值 |
|---|---|
| Base model | `$base_model` $mirror_note |
| 資料集 | [yentinglin/TaiwanChat](https://huggingface.co/datasets/yentinglin/TaiwanChat):取樣 $n_samples 筆 → 過濾超過 $max_seq tokens 後剩 $n_kept 筆 |
| 資料源(id) | $sources |
| 方法 | QLoRA:4-bit NF4 載入 + LoRA(r=$lora_r、alpha=$lora_alpha、dropout=0;q/k/v/o/gate/up/down 七個投影層) |
| max_seq_length | $max_seq |
| Learning rate | $lr(linear decay、warmup ratio 0.03) |
| 訓練量 | $train_amount(有效 batch size 8) |
| Optimizer | adamw_8bit(weight decay 0.001) |
| Loss 遮罩 | 只對 assistant 回覆計 loss(unsloth `train_on_responses_only`) |
| GPU / 時間 | $gpu / 約 $runtime_min 分鐘 |
| 最終 train loss | $final_loss |
| Seed | $seed |
| 訓練日期 | $run_date |

![training loss](loss_curve.png)

## 使用方式

$usage

## 微調前後對照

同樣 5 個台灣情境問題、greedy decoding(可重現),差異完全來自微調:

$examples

## 限制

- 事實型內容(電信資費、店家、時刻)可能過時或幻覺,請勿當作查詢工具。
- 訓練資料含翻譯/合成語料,偶爾仍會滲出簡體中文或英文片段。
- 未做額外安全對齊,安全行為同 base model。

## 授權與合規

- 模型權重依 [$license_name]($license_link) 發佈(repo 內附 LICENSE.txt 與 NOTICE),使用須遵守 [Acceptable Use Policy]($license_aup)。
- 訓練資料 TaiwanChat 為 [CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/):**本模型僅供研究/非商業用途**。

## 引用

```bibtex
@misc{lin2023taiwanllm,
  title={Taiwan LLM: Bridging the Linguistic Divide with a Culturally Aligned Language Model},
  author={Yen-Ting Lin and Yun-Nung Chen},
  year={2023},
  eprint={2311.17487},
  archivePrefix={arXiv}
}
```

訓練使用 [Unsloth](https://github.com/unslothai/unsloth)。訓練程式碼:[github.com/tun0000/llama32-taiwanchat-qlora](https://github.com/tun0000/llama32-taiwanchat-qlora)
""")

USAGE_LORA = Template("""這個 repo 只有 **LoRA adapter**(~幾十 MB),要搭配 base model 載入:

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained("$base_model", torch_dtype="auto", device_map="auto")
model = PeftModel.from_pretrained(base, "$repo")
tokenizer = AutoTokenizer.from_pretrained("$repo")

messages = [{"role": "user", "content": "請推薦台南的夜市小吃"}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
print(tokenizer.decode(model.generate(inputs, max_new_tokens=256)[0, inputs.shape[1]:], skip_special_tokens=True))
```

想直接下載完整模型(不用自己合併)→ [$merged](https://huggingface.co/$merged)。""").substitute(
    base_model=BASE_MODEL, repo=_adapter_repo, merged=_merged_repo)

USAGE_MERGED = Template("""這是**合併後的完整 fp16 模型**(~6.5 GB),可直接用 transformers 載入:

```python
from transformers import pipeline

pipe = pipeline("text-generation", model="$repo", torch_dtype="auto", device_map="auto")
out = pipe([{"role": "user", "content": "請推薦台南的夜市小吃"}], max_new_tokens=256)
print(out[0]["generated_text"][-1]["content"])
```

也可用 vLLM / Unsloth 載入。只想要輕量 adapter → [$adapter](https://huggingface.co/$adapter)。""").substitute(
    repo=_merged_repo, adapter=_adapter_repo)

_common = dict(
    base_model=BASE_MODEL, smoke_banner=SMOKE_BANNER,
    sources=("全部 9 種" if INCLUDE_SOURCES is None else ", ".join(sorted(set(INCLUDE_SOURCES)))),
    n_samples=N_SAMPLES, n_kept=prep_stats["n_kept"], max_seq=MAX_SEQ_LENGTH,
    lora_r=LORA_R, lora_alpha=LORA_ALPHA, lr=LEARNING_RATE,
    train_amount=TRAIN_AMOUNT, gpu=GPU_NAME, runtime_min=RUNTIME_MIN,
    final_loss=FINAL_LOSS, seed=SEED, run_date=RUN_DATE,
    examples=_examples_md(comparison), mirror_note=MIRROR_NOTE,
    license_slug=LICENSE_SLUG, license_name=LICENSE_NAME,
    license_link=LICENSE_LINK, license_aup=LICENSE_AUP, llama_ver=_LLAMA_VER,
    en_summary=(f"{BASE_MODEL_DISPLAY} fine-tuned with QLoRA (Unsloth) on a subset of "
                "yentinglin/TaiwanChat to improve Traditional-Chinese (Taiwan) fluency and "
                "localization. Loss was computed on assistant responses only. Research / "
                "non-commercial use only (the dataset is CC BY-NC 4.0)."),
)

card_lora = CARD_TEMPLATE.substitute(
    _common, title=f"{BASE_MODEL_DISPLAY.replace('Meta-', '').replace('-Instruct', '')} Taiwan Chat{RUN_LABEL} — LoRA adapter",
    library_name="peft", usage=USAGE_LORA,
    intro=(f"以 QLoRA(Unsloth)在 TaiwanChat 子集上微調 {BASE_MODEL_DISPLAY} 得到的 "
           f"**LoRA adapter**,目標是更自然的繁體中文與台灣在地語感。"),
)
card_merged = CARD_TEMPLATE.substitute(
    _common, title=f"{BASE_MODEL_DISPLAY.replace('Meta-', '').replace('-Instruct', '')} Taiwan Chat{RUN_LABEL}",
    library_name="transformers", usage=USAGE_MERGED,
    intro=(f"以 QLoRA(Unsloth)在 TaiwanChat 子集上微調 {BASE_MODEL_DISPLAY} 後,"
           f"**合併為完整 fp16 權重**的版本,下載即用,目標是更自然的繁體中文與台灣在地語感。"),
)
print(card_lora[:1000])
print("…\n" + "=" * 80)
print("兩份 model card 已生成(card_lora / card_merged)。")


In [ ]:
# 推送 LoRA adapter(便宜、幾秒鐘 → 先推,斷線也保得住成果)
from huggingface_hub import ModelCard

if PUSH_TO_HUB:
    model.push_to_hub(ADAPTER_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(ADAPTER_REPO, token=HF_TOKEN)          # 兩個都要推
    ModelCard(card_lora).push_to_hub(ADAPTER_REPO, token=HF_TOKEN)   # 蓋掉自動生成的 README stub
    api.upload_file(path_or_fileobj=NOTICE_TEXT.encode("utf-8"), path_in_repo="NOTICE", repo_id=ADAPTER_REPO)
    api.upload_file(path_or_fileobj="loss_curve.png", path_in_repo="loss_curve.png", repo_id=ADAPTER_REPO)
    print(f"✅ adapter → https://huggingface.co/{ADAPTER_REPO}")
else:
    model.save_pretrained("lora_out")
    tokenizer.save_pretrained("lora_out")
    print("SMOKE_TEST:不推送。adapter 已存到 lora_out/(連序列化流程也一起驗證)")


In [ ]:
# 推送 merged 16-bit 完整模型(~6.5 GB 上傳,較久)
if PUSH_TO_HUB:
    model.push_to_hub_merged(MERGED_REPO, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
    ModelCard(card_merged).push_to_hub(MERGED_REPO, token=HF_TOKEN)
    api.upload_file(path_or_fileobj=NOTICE_TEXT.encode("utf-8"), path_in_repo="NOTICE", repo_id=MERGED_REPO)
    api.upload_file(path_or_fileobj="loss_curve.png", path_in_repo="loss_curve.png", repo_id=MERGED_REPO)

    # Llama 3.2 授權要求附 LICENSE 副本:先從 base repo 拿,失敗再抓 meta 官方 GitHub;都失敗僅警告(卡片有連結)
    lic_bytes = None
    try:
        from huggingface_hub import hf_hub_download
        lic_bytes = open(hf_hub_download(BASE_MODEL, "LICENSE.txt", token=HF_TOKEN), "rb").read()
    except Exception:
        try:
            import urllib.request
            lic_bytes = urllib.request.urlopen(
                "https://raw.githubusercontent.com/meta-llama/llama-models/main/models/llama3_2/LICENSE"
            ).read()
        except Exception as e:
            print("LICENSE.txt 取得失敗,略過上傳:", e)
    if lic_bytes:
        for repo in (ADAPTER_REPO, MERGED_REPO):
            api.upload_file(path_or_fileobj=lic_bytes, path_in_repo="LICENSE.txt", repo_id=repo)
    print(f"✅ merged → https://huggingface.co/{MERGED_REPO}")
else:
    print("SMOKE_TEST:不推送 merged 模型。")


## ✅ 完成!

- LoRA adapter:`https://huggingface.co/<HF_USERNAME>/llama-3.2-3b-taiwan-chat-lora`
- 完整合併模型:`https://huggingface.co/<HF_USERNAME>/llama-3.2-3b-taiwan-chat`

**如果這次是 SMOKE_TEST:** 回到最上面的參數 cell 改 `SMOKE_TEST = False`,再 Runtime → Run all 正式訓練。

之後想匯出 GGUF(Ollama / LM Studio 用)可以加一個 cell:

```python
model.push_to_hub_gguf(f"{HF_USERNAME}/llama-3.2-3b-taiwan-chat-gguf", tokenizer,
                       quantization_method="q4_k_m", token=HF_TOKEN)
```
